# 03. 기업 리뷰 텍스트 전처리와 카테고리 신호 추출

## 목적

이 노트북은 데이터 기반 취업 지원 서비스 프로젝트에서 사용한 실제 잡플래닛 리뷰 데이터 일부와 실제 사전 파일을 기반으로, 리뷰 텍스트를 분석 가능한 카테고리 신호로 변환하는 과정을 정리한다.

사용 파일:

- `2026-04-24-11-35-48-잡플래닛.xlsx`
- `260503_복합명사등록_사전_v1.xlsx`
- `260503_카테고리사전_v1.xlsx`
- `260503_표준화_사전_최종_v1.xlsx`

## 보여주는 역량

- 실제 리뷰 원문 구조 확인
- 리뷰 제목/장점/단점/경영진 의견의 long format 변환
- 텍스트 정제 및 리뷰 길이 계산
- 표준화 사전 적용
- 실제 카테고리 사전 기반 키워드 매핑
- 장점/단점/경영진 의견 구조를 활용한 context signal 분리
- 기업 단위 카테고리 신호 요약
- 카테고리 커버리지와 미매칭 리뷰 검증

In [ ]:
import re
from collections import defaultdict
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/03_review")
OUTPUT_DIR = Path("../outputs/03_review")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

review_path = DATA_DIR / "2026-04-24-11-35-48-잡플래닛.xlsx"
compound_path = DATA_DIR / "260503_복합명사등록_사전_v1.xlsx"
category_path = DATA_DIR / "260503_카테고리사전_v1.xlsx"
standard_path = DATA_DIR / "260503_표준화_사전_최종_v1.xlsx"

reviews = pd.read_excel(review_path)
compound = pd.read_excel(compound_path)
category = pd.read_excel(category_path)
standard = pd.read_excel(standard_path)

for df in [reviews, compound, category, standard]:
    df.columns = [str(c).strip() for c in df.columns]

print("reviews:", reviews.shape)
print("compound dictionary:", compound.shape)
print("category dictionary:", category.shape)
print("standard dictionary:", standard.shape)
reviews.head()

## 1. 실제 리뷰 데이터 구조 확인

잡플래닛 리뷰 파일에는 기업명, 전체평점, 복지/급여, 워라밸, 사내문화, 승진기회, 경영진, 리뷰제목, 장점, 단점, 경영진에 바라는 점 등이 포함되어 있다.

먼저 전체 리뷰 행 수, 기업 수, 평점 컬럼, 텍스트 컬럼을 확인한다.

In [ ]:
review_schema_summary = pd.DataFrame([
    {
        "dataset": "jobplanet_reviews",
        "rows": len(reviews),
        "columns": len(reviews.columns),
        "columns_list": ", ".join(reviews.columns.astype(str).tolist())
    }
])

company_col = "기업명"
rating_cols = [c for c in ["전체평점", "복지/급여", "워라밸", "사내문화", "승진기회", "경영진"] if c in reviews.columns]
text_cols = [c for c in ["리뷰제목", "장점", "단점", "경영진에 바라는 점"] if c in reviews.columns]

print("company count:", reviews[company_col].nunique())
print("rating columns:", rating_cols)
print("text columns:", text_cols)

review_schema_summary.to_csv(OUTPUT_DIR / "03_review_schema_summary.csv", index=False, encoding="utf-8-sig")
review_schema_summary

In [ ]:
company_review_count = (
    reviews.groupby(company_col)
    .size()
    .reset_index(name="review_count")
    .sort_values("review_count", ascending=False)
)
company_review_count.head(10)

## 2. 리뷰 텍스트 long format 변환

원본 데이터는 리뷰제목, 장점, 단점, 경영진에 바라는 점이 각각 별도 컬럼이다.

이 구조를 그대로 두면 텍스트 유형별 의미를 활용하기 어렵기 때문에, 하나의 리뷰를 여러 텍스트 행으로 분해한다.

- `title`: 중립 맥락
- `pros`: 긍정 맥락
- `cons`: 부정 맥락
- `to_management`: 개선 요구 맥락

In [ ]:
reviews_work = reviews.copy()
reviews_work["review_id"] = np.arange(1, len(reviews_work) + 1)
reviews_work["company"] = reviews_work[company_col].astype(str).str.strip()

text_sources = [
    ("title", "리뷰제목", "neutral"),
    ("pros", "장점", "positive_context"),
    ("cons", "단점", "negative_context"),
    ("to_management", "경영진에 바라는 점", "improvement_context"),
]

long_rows = []
for _, row in reviews_work.iterrows():
    for text_type, col, context in text_sources:
        if col not in reviews_work.columns:
            continue
        value = row[col]
        if pd.isna(value) or str(value).strip() == "":
            continue

        out = {
            "review_id": row["review_id"],
            "company": row["company"],
            "text_type": text_type,
            "context_signal": context,
            "review_text": str(value),
        }
        for rating_col in rating_cols:
            out[rating_col] = row.get(rating_col)
        long_rows.append(out)

review_long = pd.DataFrame(long_rows)
print("original review rows:", len(reviews))
print("long format text rows:", len(review_long))
review_long.head()

## 3. 텍스트 정제

불필요한 공백, 줄바꿈, 특수문자를 정리하고 리뷰 길이를 계산한다.

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w가-힣\s.,!?%+-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

review_long["clean_text"] = review_long["review_text"].apply(clean_text)
review_long["review_length"] = review_long["clean_text"].str.len()
review_long["is_short_text"] = review_long["review_length"] < 10

cleaning_summary = pd.DataFrame([
    {"check_item": "original_review_rows", "value": len(reviews), "description": "원본 리뷰 행 수"},
    {"check_item": "long_format_text_rows", "value": len(review_long), "description": "텍스트 유형별 long format 행 수"},
    {"check_item": "empty_clean_text_rows", "value": int((review_long["clean_text"] == "").sum()), "description": "정제 후 공백 텍스트 수"},
    {"check_item": "short_text_rows_under_10_chars", "value": int(review_long["is_short_text"].sum()), "description": "10자 미만 텍스트 수"},
    {"check_item": "avg_review_length", "value": round(review_long["review_length"].mean(), 2), "description": "평균 텍스트 길이"},
    {"check_item": "median_review_length", "value": round(review_long["review_length"].median(), 2), "description": "중앙값 텍스트 길이"},
])

cleaning_summary.to_csv(OUTPUT_DIR / "03_review_cleaning_summary.csv", index=False, encoding="utf-8-sig")
cleaning_summary

## 4. 사전 구조 정리

실제 프로젝트에서 사용한 표준화 사전, 카테고리 사전, 복합명사 사전을 불러와 분석 가능한 형태로 정리한다.

사전 전체를 모델 성능처럼 주장하는 것이 아니라, 리뷰 표현을 일관된 기준으로 매핑하기 위한 전처리 기준으로 사용한다.

In [ ]:
def infer_col(df, candidates):
    cols = list(df.columns)
    normalized = {str(c).strip().lower(): c for c in cols}
    for cand in candidates:
        key = cand.strip().lower()
        if key in normalized:
            return normalized[key]
    for c in cols:
        cstr = str(c).strip().lower()
        for cand in candidates:
            if cand.strip().lower() in cstr:
                return c
    return None

def extract_standard_pairs(df):
    raw_col = infer_col(df, ["원문", "비표준", "raw", "before", "변환전", "표현"])
    std_col = infer_col(df, ["표준어", "표준", "standard", "after", "변환후", "표준표현"])
    if raw_col and std_col and raw_col != std_col:
        pairs = df[[raw_col, std_col]].dropna().copy()
        pairs.columns = ["raw_term", "standard_term"]
    else:
        use_cols = [c for c in df.columns if df[c].notna().sum() > 0]
        pairs = df[[use_cols[0], use_cols[1]]].dropna().copy()
        pairs.columns = ["raw_term", "standard_term"]
    pairs["raw_term"] = pairs["raw_term"].astype(str).str.strip()
    pairs["standard_term"] = pairs["standard_term"].astype(str).str.strip()
    return pairs.drop_duplicates()

def extract_category_terms(df):
    cat_col = infer_col(df, ["대분류", "카테고리", "category", "domain", "분류"])
    keyword_col = infer_col(df, ["단어 (토큰)", "단어", "키워드", "keyword", "token", "표현"])
    if cat_col and keyword_col and cat_col != keyword_col:
        out = df[[cat_col, keyword_col]].dropna().copy()
        out.columns = ["category", "keyword"]
    else:
        use_cols = [c for c in df.columns if df[c].notna().sum() > 0]
        out = df[[use_cols[0], use_cols[1]]].dropna().copy()
        out.columns = ["category", "keyword"]
    out["category"] = out["category"].astype(str).str.strip()
    out["keyword"] = out["keyword"].astype(str).str.strip()
    return out.drop_duplicates()

def extract_compound_terms(df):
    term_col = infer_col(df, ["복합명사", "단어", "키워드", "term", "token", "명사"])
    if term_col:
        out = df[[term_col]].dropna().copy()
        out.columns = ["compound_term"]
    else:
        use_cols = [c for c in df.columns if df[c].notna().sum() > 0]
        out = df[[use_cols[0]]].dropna().copy()
        out.columns = ["compound_term"]
    out["compound_term"] = out["compound_term"].astype(str).str.strip()
    return out.drop_duplicates()

standard_pairs = extract_standard_pairs(standard)
category_terms = extract_category_terms(category)
compound_terms = extract_compound_terms(compound)

standard_pairs.to_csv(DATA_DIR / "standard_dictionary_normalized.csv", index=False, encoding="utf-8-sig")
category_terms.to_csv(DATA_DIR / "category_dictionary_normalized.csv", index=False, encoding="utf-8-sig")
compound_terms.to_csv(DATA_DIR / "compound_noun_dictionary_normalized.csv", index=False, encoding="utf-8-sig")

dictionary_summary = pd.DataFrame([
    {"dictionary": "standard_dictionary", "rows": len(standard_pairs), "columns": "raw_term, standard_term"},
    {"dictionary": "category_dictionary", "rows": len(category_terms), "columns": "category, keyword"},
    {"dictionary": "compound_noun_dictionary", "rows": len(compound_terms), "columns": "compound_term"},
])
dictionary_summary.to_csv(OUTPUT_DIR / "03_dictionary_summary.csv", index=False, encoding="utf-8-sig")
dictionary_summary

## 5. 표준화 사전 적용

표준화 사전은 같은 의미의 표현을 하나의 기준 표현으로 맞추기 위해 사용한다.

예를 들어 `워라벨`, `워라밸`처럼 표기가 다른 표현을 통일하면 이후 카테고리 매핑의 누락을 줄일 수 있다.

In [ ]:
standard_map = dict(zip(standard_pairs["raw_term"], standard_pairs["standard_term"]))

def apply_standardization(text, max_pairs=5000):
    if not isinstance(text, str):
        return ""
    result = text
    for raw_term, standard_term in sorted(standard_map.items(), key=lambda x: len(str(x[0])), reverse=True)[:max_pairs]:
        if raw_term and raw_term != "nan":
            result = result.replace(str(raw_term), str(standard_term))
    return result

review_long["standardized_text"] = review_long["clean_text"].apply(apply_standardization)

review_long.to_csv(OUTPUT_DIR / "03_review_text_long_format.csv", index=False, encoding="utf-8-sig")
review_long[["review_id", "company", "text_type", "context_signal", "clean_text", "standardized_text"]].head()

## 6. 카테고리 사전 기반 매핑

카테고리 사전의 키워드가 리뷰 텍스트에 포함되어 있는지 확인해 리뷰별 카테고리 신호를 만든다.

한 리뷰 텍스트는 여러 카테고리에 동시에 매핑될 수 있다.

In [ ]:
category_dict = defaultdict(list)
for _, row in category_terms.iterrows():
    category_dict[row["category"]].append(row["keyword"])

categories = sorted(category_dict.keys())

def match_categories(text):
    matched_categories = []
    matched_keywords = []
    for cat, keywords in category_dict.items():
        hits = [kw for kw in keywords if kw and kw in text]
        if hits:
            matched_categories.append(cat)
            matched_keywords.extend([f"{cat}:{kw}" for kw in hits[:5]])
    return sorted(set(matched_categories)), matched_keywords

mapping_rows = []
for _, row in review_long.iterrows():
    matched_categories, matched_keywords = match_categories(row["standardized_text"])
    out = row.to_dict()
    out["matched_category_count"] = len(matched_categories)
    out["matched_categories"] = "|".join(matched_categories)
    out["matched_keywords"] = "|".join(matched_keywords[:30])
    out["has_category_match"] = len(matched_categories) > 0

    for cat in categories:
        out[f"cat_{cat}"] = int(cat in matched_categories)

    mapping_rows.append(out)

mapping_result = pd.DataFrame(mapping_rows)
mapping_result.to_csv(OUTPUT_DIR / "03_review_category_mapping_result.csv", index=False, encoding="utf-8-sig")
mapping_result.head()

## 7. 기업 단위 카테고리 신호 요약

리뷰 텍스트 단위 결과를 기업 단위로 집계한다.

이 결과는 기업별 리뷰 신호를 비교하고, 추천/비추천 판단 로직에 활용할 수 있는 입력 테이블이 된다.

In [ ]:
flag_cols = [col for col in mapping_result.columns if col.startswith("cat_")]

agg_dict = {
    "review_id": "nunique",
    "clean_text": "count",
    "has_category_match": "sum",
    "review_length": "mean",
}
for col in flag_cols:
    agg_dict[col] = "sum"

company_summary = mapping_result.groupby("company").agg(agg_dict).reset_index()
company_summary = company_summary.rename(columns={
    "review_id": "review_count",
    "clean_text": "text_segment_count",
    "has_category_match": "matched_text_segment_count",
    "review_length": "avg_text_length"
})
company_summary["category_match_rate"] = (
    company_summary["matched_text_segment_count"] / company_summary["text_segment_count"] * 100
).round(2)
company_summary["avg_text_length"] = company_summary["avg_text_length"].round(2)

context_pivot = (
    mapping_result.groupby(["company", "context_signal"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
company_summary = company_summary.merge(context_pivot, on="company", how="left")

company_summary.to_csv(OUTPUT_DIR / "03_company_category_signal_summary.csv", index=False, encoding="utf-8-sig")
company_summary.head()

## 8. 카테고리 커버리지 검증

사전 기반 방식은 설명 가능성이 높지만, 사전에 없는 표현은 놓칠 수 있다.  
따라서 전체 텍스트 중 얼마나 많은 텍스트가 카테고리에 매칭되었는지 확인한다.

In [ ]:
coverage_summary = pd.DataFrame([
    {"metric": "raw_review_rows", "value": len(reviews), "description": "원본 리뷰 행 수"},
    {"metric": "company_count", "value": reviews_work["company"].nunique(), "description": "기업 수"},
    {"metric": "text_segment_rows", "value": len(review_long), "description": "title/pros/cons/to_management 분해 후 텍스트 행 수"},
    {"metric": "short_text_count", "value": int(review_long["is_short_text"].sum()), "description": "10자 미만 텍스트 수"},
    {"metric": "matched_text_segment_count", "value": int(mapping_result["has_category_match"].sum()), "description": "카테고리 1개 이상 매칭 텍스트 수"},
    {"metric": "unmatched_text_segment_count", "value": int((~mapping_result["has_category_match"]).sum()), "description": "카테고리 미매칭 텍스트 수"},
    {"metric": "category_match_rate_percent", "value": round(mapping_result["has_category_match"].mean() * 100, 2), "description": "카테고리 1개 이상 매칭 비율"},
    {"metric": "standard_dictionary_terms", "value": len(standard_pairs), "description": "표준화 사전 항목 수"},
    {"metric": "category_dictionary_terms", "value": len(category_terms), "description": "카테고리 사전 항목 수"},
    {"metric": "compound_noun_terms", "value": len(compound_terms), "description": "복합명사 사전 항목 수"},
])

coverage_summary.to_csv(OUTPUT_DIR / "03_category_coverage_summary.csv", index=False, encoding="utf-8-sig")
coverage_summary

In [ ]:
category_mention_summary = []
for col in flag_cols:
    cat = col.replace("cat_", "", 1)
    category_mention_summary.append({
        "category": cat,
        "mention_text_segment_count": int(mapping_result[col].sum()),
        "mention_rate_percent": round(mapping_result[col].mean() * 100, 2)
    })

category_mention_summary = (
    pd.DataFrame(category_mention_summary)
    .sort_values("mention_text_segment_count", ascending=False)
)
category_mention_summary.to_csv(OUTPUT_DIR / "03_category_mention_summary.csv", index=False, encoding="utf-8-sig")
category_mention_summary.head(20)

In [ ]:
unmatched_sample = mapping_result.loc[
    ~mapping_result["has_category_match"],
    ["review_id", "company", "text_type", "context_signal", "clean_text", "review_length"]
].head(100)

unmatched_sample.to_csv(OUTPUT_DIR / "03_unmatched_review_sample.csv", index=False, encoding="utf-8-sig")
unmatched_sample.head()

## 최종 요약

이 노트북은 실제 기업 리뷰 원문을 분석 가능한 카테고리 신호로 바꾸는 전처리 과정을 보여준다.

핵심은 다음과 같다.

1. 잡플래닛 리뷰의 `장점`, `단점`, `경영진에 바라는 점` 구조를 활용해 텍스트 맥락을 구분했다.
2. 표준화 사전을 적용해 표현 차이로 인한 카테고리 매핑 누락을 줄였다.
3. 실제 카테고리 사전을 기준으로 리뷰 텍스트를 복수 카테고리에 매핑했다.
4. 리뷰 단위 신호를 기업 단위 요약 테이블로 집계했다.
5. 카테고리 매칭률과 미매칭 샘플을 확인해 사전 기반 방식의 커버리지와 한계를 검증했다.

단, 이 방식은 사전 기반 매핑이므로 문맥 반전, 비꼼, 사전에 없는 표현을 완벽히 처리하지 못한다.  
따라서 이 결과는 기업 평가를 자동 확정하는 지표가 아니라, 리뷰 신호를 구조화해 추천/비추천 판단에 참고하는 보조 지표로 활용해야 한다.